In [1]:
import numpy as np
import pandas as pd
from mhcgnomes import parse

# 1 Read total data

In [53]:
#DATA_PATH = '/home/denklewer/Data/' #ubuntu
EXPERIMENT_NAME = 'length_dependent_hmm/immunogenic'
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/' # windows

IMMUNOGENICITY_PRIME_DATA_PATH = DATA_PATH + 'immunogenicity/PRIME/1-s2.0-S2666379121000057-mmc2.csv'


In [54]:
imm_df = pd.read_csv(IMMUNOGENICITY_PRIME_DATA_PATH, sep=';')


In [55]:
len(imm_df)

7758

In [56]:
imm_df[['Allele', 'Immunogenicity']].value_counts()

Allele  Immunogenicity
A0201   0                 868
        1                 658
A1101   0                 586
B1501   0                 287
B0702   0                 275
                         ... 
B4901   1                   1
C1403   1                   1
B3701   1                   1
C1502   1                   1
B5601   1                   1
Length: 113, dtype: int64

In [57]:
imm_df

,Mutant,Allele,MixMHCpred,NetMHCpanEL,NetMHCpanBA,NetMHCpanBA_Kd,NetMHCstabpan,NetMHCstabpan_T12,MHCflurry,HLAthena,...,WT_peptide,NetMHCpanEL_WT,NetMHCpanBA_Kd_WT,ratio_%rank,ratio_Kd,DisToSelf,DisToSelf_peptide,Immunogenicity,StudyOrigin,PRIME
0,VMLQAPLFT,A0201,7.00,2.391,2.193,356.05,8.50,0.54,0.85350,8.40544,...,DMLQAPLFT,12.165,10345.08,1.626851,0.034417,9,DMLQAPLFT,0,Bobisse,0.000000
1,MLIVETAVM,A0201,7.00,1.936,1.610,206.86,9.00,0.53,1.33500,8.40110,...,MLIVETADM,3.475,569.10,0.584970,0.363486,9,MLIVETADM,0,Bobisse,0.000000
2,MLIVETAVML,A0201,8.00,1.256,0.586,47.33,3.00,1.39,0.81200,3.18349,...,MLIVETADML,2.789,140.70,0.797751,0.336389,9,MLIVETADML,0,Bobisse,0.000000
3,VMLQAPLFTA,A0201,2.00,1.562,0.572,45.70,1.60,2.41,0.22150,0.76095,...,DMLQAPLFTA,5.188,221.03,1.200381,0.206759,9,DMLQAPLFTA,0,Bobisse,0.174395
4,LLSECVHAL,A0201,0.01,0.067,0.045,5.75,0.08,14.09,0.02350,0.18895,...,LVSECVHAL,0.680,100.73,2.317400,0.057083,3,LVSECVHAL,0,Bobisse,0.205919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7753,QNQQQCQPP,C1601,79.00,80.000,79.017,42117.54,70.00,0.09,52.94125,97.18529,...,NaN,NaN,NaN,NaN,NaN,0,QNQQQCQPP,0,Random,0.000000
7754,KTQEAQVLK,C1601,11.00,10.388,19.237,15478.82,12.00,0.19,3.84788,42.31253,...,NaN,NaN,NaN,NaN,NaN,0,KTQEAQVLK,0,Random,0.000000
7755,PLCSRVFPS,C1601,84.00,100.000,77.122,41617.89,70.00,0.10,45.32062,76.53923,...,NaN,NaN,NaN,NaN,NaN,0,PLCSRVFPS,0,Random,0.000000
7756,GGAIKELSS,C1601,11.00,53.000,61.672,37165.36,70.00,0.08,31.37450,53.74716,...,NaN,NaN,NaN,NaN,NaN,0,GGAIKELSS,0,Random,0.000000


In [58]:
imm_df.columns

Index(['Mutant', 'Allele', 'MixMHCpred', 'NetMHCpanEL', 'NetMHCpanBA',
       'NetMHCpanBA_Kd', 'NetMHCstabpan', 'NetMHCstabpan_T12', 'MHCflurry',
       'HLAthena', 'NetChop', 'TAP', 'IEDB', 'Foreignness', 'WT_peptide',
       'NetMHCpanEL_WT', 'NetMHCpanBA_Kd_WT', 'ratio_%rank', 'ratio_Kd',
       'DisToSelf', 'DisToSelf_peptide', 'Immunogenicity', 'StudyOrigin',
       'PRIME'],
      dtype='object')

In [59]:
from mhcgnomes import parse

In [60]:
ALLELES = ['HLA-A*02:01']

In [61]:
from mhcgnomes import Allele

In [62]:
def tuple_to_last_elem(column):
    return column[-1]

def filter_immunogenicity_dataset(imm_df):

    
    print("Loaded immunogenicity data: %s" % str(imm_df.shape))
    filtered_df = imm_df.copy()
    
    ## Clear bad data
    print("Subselecting to valid peptides (dropping peptides with PTMs). Starting with: %d" % len(filtered_df))
    filtered_df['Mutant'] = filtered_df['Mutant'].str.strip()
    filtered_df = filtered_df.loc[
        imm_df['Mutant'].str.match("^[ACDEFGHIKLMNPQRSTVWY]+$")
    ]
    print("Now: %d" % len(filtered_df))
    
    
    filtered_df = filtered_df.rename(columns={"Allele":"allele", 
                                              "Mutant":"peptide",
                                             "Immunogenicity":"immunogenic"})
    filtered_df['allele'] = filtered_df['allele'].apply(str).map(parse).apply(Allele.to_string)
    filtered_df['length'] = filtered_df['peptide'].str.len()
    
    #filtered_df['allele'] = filtered_df['original_allele'].map(parse).to_string()
    
    filtered_df = filtered_df.drop_duplicates([
    'peptide',
    'allele',
    'length',
    'immunogenic']).reset_index(drop=True)
    print("Now: %d" % len(filtered_df))

    
    print("Allele_distribution")
    print(filtered_df['allele'].value_counts()[:30])
    

    
    assert 'peptide' in filtered_df.columns
    assert 'allele' in filtered_df.columns
    assert 'length' in filtered_df.columns
    assert 'immunogenic' in filtered_df.columns
    return filtered_df

In [63]:
imm_df = filter_immunogenicity_dataset(imm_df)
    

Loaded immunogenicity data: (7758, 24)
Subselecting to valid peptides (dropping peptides with PTMs). Starting with: 7758
Now: 7758
Now: 7758
Allele_distribution
HLA-A*02:01    1526
HLA-A*11:01     670
HLA-B*07:02     342
HLA-B*15:01     295
HLA-B*35:01     266
HLA-A*01:01     250
HLA-A*24:02     245
HLA-A*32:01     183
HLA-A*03:01     182
HLA-A*31:01     179
HLA-C*03:03     165
HLA-A*02:06     131
HLA-A*26:01     131
HLA-B*08:01     124
HLA-A*68:01     115
HLA-B*27:05     115
HLA-A*33:01     103
HLA-B*40:01      98
HLA-A*02:11      98
HLA-A*23:01      97
HLA-B*44:03      96
HLA-A*30:01      96
HLA-C*06:02      86
HLA-B*44:02      82
HLA-B*57:01      80
HLA-B*58:01      79
HLA-B*18:01      78
HLA-C*07:01      77
HLA-B*45:01      74
HLA-A*68:02      74
Name: allele, dtype: int64


In [64]:
imm_df

,peptide,allele,MixMHCpred,NetMHCpanEL,NetMHCpanBA,NetMHCpanBA_Kd,NetMHCstabpan,NetMHCstabpan_T12,MHCflurry,HLAthena,...,NetMHCpanEL_WT,NetMHCpanBA_Kd_WT,ratio_%rank,ratio_Kd,DisToSelf,DisToSelf_peptide,immunogenic,StudyOrigin,PRIME,length
0,VMLQAPLFT,HLA-A*02:01,7.00,2.391,2.193,356.05,8.50,0.54,0.85350,8.40544,...,12.165,10345.08,1.626851,0.034417,9,DMLQAPLFT,0,Bobisse,0.000000,9
1,MLIVETAVM,HLA-A*02:01,7.00,1.936,1.610,206.86,9.00,0.53,1.33500,8.40110,...,3.475,569.10,0.584970,0.363486,9,MLIVETADM,0,Bobisse,0.000000,9
2,MLIVETAVML,HLA-A*02:01,8.00,1.256,0.586,47.33,3.00,1.39,0.81200,3.18349,...,2.789,140.70,0.797751,0.336389,9,MLIVETADML,0,Bobisse,0.000000,10
3,VMLQAPLFTA,HLA-A*02:01,2.00,1.562,0.572,45.70,1.60,2.41,0.22150,0.76095,...,5.188,221.03,1.200381,0.206759,9,DMLQAPLFTA,0,Bobisse,0.174395,10
4,LLSECVHAL,HLA-A*02:01,0.01,0.067,0.045,5.75,0.08,14.09,0.02350,0.18895,...,0.680,100.73,2.317400,0.057083,3,LVSECVHAL,0,Bobisse,0.205919,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7753,QNQQQCQPP,HLA-C*16:01,79.00,80.000,79.017,42117.54,70.00,0.09,52.94125,97.18529,...,NaN,NaN,NaN,NaN,0,QNQQQCQPP,0,Random,0.000000,9
7754,KTQEAQVLK,HLA-C*16:01,11.00,10.388,19.237,15478.82,12.00,0.19,3.84788,42.31253,...,NaN,NaN,NaN,NaN,0,KTQEAQVLK,0,Random,0.000000,9
7755,PLCSRVFPS,HLA-C*16:01,84.00,100.000,77.122,41617.89,70.00,0.10,45.32062,76.53923,...,NaN,NaN,NaN,NaN,0,PLCSRVFPS,0,Random,0.000000,9
7756,GGAIKELSS,HLA-C*16:01,11.00,53.000,61.672,37165.36,70.00,0.08,31.37450,53.74716,...,NaN,NaN,NaN,NaN,0,GGAIKELSS,0,Random,0.000000,9


# Make per allele DF

In [65]:
def make_per_allele_df(source_df, ALLELES):
    per_allele_df = dict()
    for allele_name in ALLELES:
        allele_df = source_df[source_df['allele'] == allele_name] 
        print(f"------Allele {allele_name} ----------")
        print("Immunogenic_distribution:")
        print(allele_df['immunogenic'].value_counts())
        print("Length_distribution:")
        print(allele_df['length'].value_counts()[:10])
        per_allele_df[allele_name] = allele_df
    return per_allele_df
        
        
per_allele_df = make_per_allele_df(imm_df, ALLELES)    

------Allele HLA-A*02:01 ----------
Immunogenic_distribution:
0    868
1    658
Name: immunogenic, dtype: int64
Length_distribution:
9     938
10    500
11     74
8      14
Name: length, dtype: int64


#  Save per allele DF

In [67]:
import os
if not os.path.exists(DATA_PATH + EXPERIMENT_NAME + '/' ):
    os.makedirs(DATA_PATH + EXPERIMENT_NAME + '/' )

In [68]:
TARGET_PATH_TO_FILES = DATA_PATH + EXPERIMENT_NAME + '/' 

for allele_name in ALLELES:
    total_df = per_allele_df[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}data_{}.csv".format(TARGET_PATH_TO_FILES,allele_name),sep=";", index=False)